In [1]:
import torch.nn as nn
import torch


class AlexNet(nn.Module):
    def __init__(self, num_classes=40, init_weights=False):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 48, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2),
            nn.Conv1d(48, 128, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2),
            nn.Conv1d(128, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(192, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(192, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(5888, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048, num_classes),
        )
        if init_weights:
            self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [2]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import numpy as np

def plot_matrix(conf_matrix, dev_list, save_path):
    plt.figure(figsize=(20, 16), dpi=300)
    plt.imshow(conf_matrix, cmap=plt.cm.Blues, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()

    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, conf_matrix[i, j],
                     ha="center", va="center",
                     color="white" if conf_matrix[i, j] > thresh else "black", fontsize=8)

    tick_marks = np.arange(len(dev_list))
    plt.xticks(tick_marks, dev_list, fontsize=8)
    plt.yticks(tick_marks, dev_list, fontsize=8)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    plt.gca().set_aspect(aspect='equal')

    plt.savefig(save_path)
    plt.show()

In [3]:
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import confusion_matrix

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.categories = sorted(os.listdir(data_dir))
        self.data = []
        self.transform = transform
        for category in self.categories:
            category_dir = os.path.join(data_dir, category)
            category_data = sorted(os.listdir(category_dir))
            self.data.extend([(os.path.join(category_dir, file), self.categories.index(category)) for file in category_data])

    def __getitem__(self, index):
        file_path, label = self.data[index]
        data = np.load(file_path)
        image = Image.fromarray(data.astype(np.uint8))
        image = transform(image)
        return image, label

    def __len__(self):
        return len(self.data)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("using {} device.".format(device))

transform = transforms.Compose([ transforms.Grayscale(num_output_channels = 1),
                                     transforms.ToTensor(),
                                     transforms.Normalize((0.5), (0.5))])

train_dataset = CustomDataset("features/train_npy",transform=transform)
train_num = len(train_dataset)
dev_list = train_dataset.categories

batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size, shuffle=True,
                                               num_workers=0)

validate_dataset = CustomDataset("features/val_npy",transform=transform)
val_num = len(validate_dataset)
validate_loader = torch.utils.data.DataLoader(validate_dataset,
                                                  batch_size=batch_size, shuffle=False,
                                                  num_workers=0)

print("using {} images for training, {} images for validation.".format(train_num, val_num))

net = AlexNet(num_classes=31, init_weights=True)
net.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

epochs = 100
save_path = './AlexNet_parameters.pth'
best_f1 = 0.0
train_accurate_list = []
val_accurate_list = []
f1_list = []
recall_list = []

for epoch in range(epochs):
    net.train()
    running_loss = 0.0
    train_acc = 0.0
    for step, data in enumerate(train_loader, start=0):
        images, labels = data
        images = images.reshape(images.shape[0], 1, 1503)
        optimizer.zero_grad()
        outputs = net(images.to(device))
        predict_y = torch.max(outputs, dim=1)[1]
        train_acc += torch.eq(predict_y, labels.to(device)).sum().item()
        loss = loss_function(outputs,labels.to(device))
        loss.backward()
        optimizer.step()


        running_loss += loss.item()
        rate = (step + 1) / len(train_loader)
        a = "*" * int(rate * 50)
        b = "." * int((1 - rate) * 50)
        print("\rtrain loss:{:^3.0f}%[{}->{}]{:.3f}".format(int(rate * 100), a, b, loss), end="")
    print()
    train_accurate = train_acc / train_num
    train_accurate_list.append(train_accurate)

    net.eval()
    acc = 0.0  
    val = torch.tensor([])
    pre = torch.tensor([])
    with torch.no_grad():
        for val_data in validate_loader:
            val_images, val_labels = val_data
            val_images = val_images.reshape(val_images.shape[0], 1, 1503)
            outputs = net(val_images.to(device))
            predict_y = torch.max(outputs, dim=1)[1]
            pre = torch.cat([pre.to(device), predict_y.to(device)])
            val = torch.cat([val.to(device), val_labels.to(device)])
            acc += torch.eq(predict_y, val_labels.to(device)).sum().item()
    val_accurate = acc / val_num
    val_accurate_list.append(val_accurate)
    f1 = f1_score(val.cpu(), pre.cpu(), average='macro')
    recall = recall_score(val.cpu(), pre.cpu(), average='macro')
    f1_list.append(f1)
    recall_list.append(recall)
    if f1 > best_f1:
        best_f1 = f1
        best_pre = pre
        best_val = val
        torch.save(net.state_dict(), save_path)
        torch.save(best_pre, 'pre_val_label/best_pre_AlexNet.pt')
        torch.save(best_val, 'pre_val_label/best_val_AlexNet.pt')
    print('[epoch %d] train_loss: %.3f train_accuracy: %.3f val_accuracy: %.3f  recall: %.3f  f1: %.3f' %
              (epoch + 1, running_loss / step, train_accurate, val_accurate, recall, f1))
    with open("AlexNet_result_npy.txt", 'a') as file:
        file.write("[epoch " + str(epoch + 1) + "]" + "  " + "train_accuracy:" + str(train_accurate) + "  " + "val_accuracy:" + str(val_accurate) + "  " + "recall:" + str(recall) + "  " + "f1:" + str(f1) + '\n')
print('Finished Training')

iterations = range(1, len(train_accurate_list) + 1)
with open("AlexNet_npy_plt_data.txt", 'a') as file:
    file.write("iterations:" + str(iterations) +
               "train_accurate_list:" + str(train_accurate_list) +
               "val_accurate_list:" + str(val_accurate_list) +
               "f1_list:" + str(f1_list) +
               "recall_list:" + str(recall_list) +
               "dev_list:" + str(dev_list) + '\n')
conf_matrix = confusion_matrix(best_val.cpu(),best_pre.cpu())
plot_matrix(conf_matrix,dev_list,"AlexNet_confusion_matrix_npy.png")

using cuda:0 device.
using 82841 images for training, 20695 images for validation.
train loss:100%[**************************************************->]1.017
[epoch 1] train_loss: 1.132 train_accuracy: 0.582 val_accuracy: 0.693  recall: 0.541  f1: 0.527
train loss:100%[**************************************************->]0.896
[epoch 2] train_loss: 0.740 train_accuracy: 0.699 val_accuracy: 0.724  recall: 0.593  f1: 0.587
train loss:100%[**************************************************->]0.762
[epoch 3] train_loss: 0.646 train_accuracy: 0.740 val_accuracy: 0.770  recall: 0.631  f1: 0.633
train loss:100%[**************************************************->]0.776
[epoch 4] train_loss: 0.569 train_accuracy: 0.775 val_accuracy: 0.799  recall: 0.671  f1: 0.681
train loss:100%[**************************************************->]0.422
[epoch 5] train_loss: 0.516 train_accuracy: 0.798 val_accuracy: 0.807  recall: 0.676  f1: 0.683
train loss:100%[**********************************************

train loss:100%[**************************************************->]0.711
[epoch 48] train_loss: 0.362 train_accuracy: 0.904 val_accuracy: 0.867  recall: 0.726  f1: 0.750
train loss:100%[**************************************************->]0.445
[epoch 49] train_loss: 0.265 train_accuracy: 0.918 val_accuracy: 0.924  recall: 0.798  f1: 0.805
train loss:100%[**************************************************->]0.313
[epoch 50] train_loss: 0.240 train_accuracy: 0.928 val_accuracy: 0.915  recall: 0.794  f1: 0.809
train loss:100%[**************************************************->]0.4510
[epoch 51] train_loss: 0.426 train_accuracy: 0.904 val_accuracy: 0.927  recall: 0.822  f1: 0.841
train loss:100%[**************************************************->]0.871
[epoch 52] train_loss: 0.319 train_accuracy: 0.909 val_accuracy: 0.901  recall: 0.772  f1: 0.778
train loss:100%[**************************************************->]0.038
[epoch 53] train_loss: 0.305 train_accuracy: 0.916 val_accuracy:

train loss:100%[**************************************************->]0.7175
[epoch 96] train_loss: 0.834 train_accuracy: 0.820 val_accuracy: 0.771  recall: 0.547  f1: 0.567
train loss:100%[**************************************************->]0.6372
[epoch 97] train_loss: 0.683 train_accuracy: 0.831 val_accuracy: 0.779  recall: 0.512  f1: 0.548
train loss:100%[**************************************************->]0.609
[epoch 98] train_loss: 0.851 train_accuracy: 0.812 val_accuracy: 0.780  recall: 0.493  f1: 0.514
train loss:100%[**************************************************->]0.299
[epoch 99] train_loss: 0.708 train_accuracy: 0.824 val_accuracy: 0.841  recall: 0.565  f1: 0.586
train loss:100%[**************************************************->]0.650
[epoch 100] train_loss: 0.797 train_accuracy: 0.811 val_accuracy: 0.814  recall: 0.543  f1: 0.561
Finished Training
